# ToxACoL Parquet Pipeline

This notebook processes `Origin/ToxACoL` CSV slice outputs in three independent stages:

1. Convert each `BatchE0XX/` folder into one batch parquet.
2. Vertically merge all batch parquets into `ToxACoL_All.parquet`.
3. Deduplicate a parquet file by `SMILES` into `ToxACoL_All_dedup.parquet`.

Each section is standalone and can be run without running the previous sections. Original CSV files are not modified.


## Convert Each Batch Folder To Parquet

Run this cell to convert every `BatchE0XX` folder into one parquet file. Set `RUN_BATCH_CONVERT = True` before running.


In [3]:
from pathlib import Path
import re

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

RUN_BATCH_CONVERT = True
OVERWRITE_BATCH_PARQUETS = False

TOXACOL_SOURCE_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL")
TOXACOL_PARQUET_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet")

# Set to None to process all batches, or choose a subset such as ["BatchE003", "BatchE004"].
BATCH_CODES = None

TOXACOL_COLUMNS = ["SMILES", "pred_mouse_intraperitoneal_LD50", "smiles_valid"]
TARGET_SCHEMA = pa.schema([
    pa.field("SMILES", pa.large_string()),
    pa.field("pred_mouse_intraperitoneal_LD50", pa.float64()),
    pa.field("smiles_valid", pa.bool_()),
])
SLICE_RE = re.compile(r"^(Batch[EG]\d{3})Slice(\d+)\.csv$", re.IGNORECASE)


def slice_key(path: Path) -> int:
    match = SLICE_RE.match(path.name)
    if not match:
        raise ValueError(f"Unexpected slice filename: {path}")
    return int(match.group(2))


def discover_batch_dirs(source_dir: Path) -> list[Path]:
    if not source_dir.exists():
        raise FileNotFoundError(f"Source directory does not exist: {source_dir}")
    return sorted(p for p in source_dir.iterdir() if p.is_dir() and p.name.startswith("Batch"))


def discover_csv_slices(batch_dir: Path) -> list[Path]:
    return sorted(batch_dir.glob("*.csv"), key=slice_key)


def validate_csv_header(path: Path) -> None:
    header = pd.read_csv(path, nrows=0).columns.tolist()
    missing = [column for column in TOXACOL_COLUMNS if column not in header]
    extra = [column for column in header if column not in TOXACOL_COLUMNS]
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")
    if extra:
        raise ValueError(f"{path} has unexpected columns: {extra}")


def align_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    missing = [column for column in TOXACOL_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(f"CSV data is missing columns: {missing}")
    out = df[TOXACOL_COLUMNS].copy()
    out["SMILES"] = out["SMILES"].astype("string")
    out["pred_mouse_intraperitoneal_LD50"] = pd.to_numeric(out["pred_mouse_intraperitoneal_LD50"], errors="coerce")
    out["smiles_valid"] = out["smiles_valid"].astype("boolean")
    return out


def selected_batch_dirs() -> list[Path]:
    batch_dirs = discover_batch_dirs(TOXACOL_SOURCE_DIR)
    if BATCH_CODES is None:
        return batch_dirs
    lookup = {p.name: p for p in batch_dirs}
    missing = [code for code in BATCH_CODES if code not in lookup]
    if missing:
        raise ValueError(f"Requested batches do not exist: {missing}")
    return [lookup[code] for code in BATCH_CODES]


def convert_batch_dir_to_parquet(batch_dir: Path, output_dir: Path, overwrite: bool = False) -> dict:
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{batch_dir.name}.parquet"
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"Output already exists: {output_path}. Set OVERWRITE_BATCH_PARQUETS = True to replace it.")

    temp_path = output_path.with_name(output_path.stem + ".tmp.parquet")
    if temp_path.exists():
        if overwrite:
            temp_path.unlink()
        else:
            raise FileExistsError(f"Temporary output already exists: {temp_path}")

    csv_files = discover_csv_slices(batch_dir)
    if not csv_files:
        raise RuntimeError(f"No CSV files found in {batch_dir}")
    validate_csv_header(csv_files[0])

    writer = None
    rows_written = 0
    try:
        writer = pq.ParquetWriter(
            temp_path,
            TARGET_SCHEMA,
            compression="snappy",
            use_dictionary=["SMILES"],
        )
        slice_iter = tqdm(csv_files, desc=batch_dir.name, unit="slice", leave=False, dynamic_ncols=True)
        for csv_path in slice_iter:
            df = align_dataframe(pd.read_csv(csv_path))
            table = pa.Table.from_pandas(df, schema=TARGET_SCHEMA, preserve_index=False)
            writer.write_table(table)
            rows_written += table.num_rows
            slice_iter.set_postfix(rows=f"{rows_written:,}")
    finally:
        if writer is not None:
            writer.close()

    if rows_written == 0:
        temp_path.unlink(missing_ok=True)
        raise RuntimeError(f"No rows were written for {batch_dir.name}")

    if output_path.exists() and overwrite:
        output_path.unlink()
    temp_path.replace(output_path)
    return {"batch": batch_dir.name, "rows": rows_written, "output_path": output_path}


print(f"ToxACoL source:     {TOXACOL_SOURCE_DIR}")
print(f"Batch parquet dir:  {TOXACOL_PARQUET_DIR}")
print(f"Selected batches:   {'all' if BATCH_CODES is None else BATCH_CODES}")

if RUN_BATCH_CONVERT:
    results = []
    batch_iter = tqdm(selected_batch_dirs(), desc="Convert ToxACoL batches", unit="batch", dynamic_ncols=True)
    for batch_dir in batch_iter:
        results.append(convert_batch_dir_to_parquet(batch_dir, TOXACOL_PARQUET_DIR, overwrite=OVERWRITE_BATCH_PARQUETS))
    print("Batch conversion complete.")
    print(f"Batch outputs: {len(results)}")
    print(f"Total rows:    {sum(item['rows'] for item in results):,}")
    for item in results:
        print(f"{item['batch']}: rows={item['rows']:,} output={item['output_path']}")
else:
    batch_dirs = selected_batch_dirs()
    total_csv = 0
    for batch_dir in batch_dirs:
        csv_files = discover_csv_slices(batch_dir)
        total_csv += len(csv_files)
        print(f"{batch_dir.name}: {len(csv_files)} csv slice(s)")
    print("-" * 60)
    print("Dry run only. Set RUN_BATCH_CONVERT = True to create batch parquet files.")
    print(f"Batches:   {len(batch_dirs)}")
    print(f"CSV files: {total_csv}")


ToxACoL source:     C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL
Batch parquet dir:  C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet
Selected batches:   all


Convert ToxACoL batches:   0%|          | 0/22 [00:00<?, ?batch/s]

BatchE003:   0%|          | 0/100 [00:00<?, ?slice/s]

BatchE004:   0%|          | 0/100 [00:00<?, ?slice/s]

BatchE005:   0%|          | 0/100 [00:00<?, ?slice/s]

...

BatchE021:   0%|          | 0/21 [00:00<?, ?slice/s]

BatchE022:   0%|          | 0/1 [00:00<?, ?slice/s]

BatchE023:   0%|          | 0/2 [00:00<?, ?slice/s]

BatchE024:   0%|          | 0/7 [00:00<?, ?slice/s]

Batch conversion complete.
Batch outputs: 22
Total rows:    8,716,446
BatchE003: rows=499,997 output=C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\BatchE003.parquet
BatchE004: rows=499,996 output=C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\BatchE004.parquet
BatchE005: rows=499,996 output=C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\BatchE005.parquet
BatchE006: rows=63,172 output=C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\BatchE006.parquet
BatchE007: rows=619,894 output=C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\BatchE007.parquet
BatchE008: rows=309,224 output=C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\BatchE008.parquet
BatchE009: rows=240,691 output=C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\BatchE009.parquet
BatchE010: rows=719,995 output=C:\Users\Cenking\Documents\ExperimentData\DryD

## Merge Batch Parquets Into One Parquet

Run this cell to vertically merge all `Batch*.parquet` files into `ToxACoL_All.parquet`. Set `RUN_MERGE = True` before running.


In [4]:
from pathlib import Path
import math

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

RUN_MERGE = True
OVERWRITE_ALL_OUTPUT = False

TOXACOL_PARQUET_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet")
ALL_OUTPUT_PATH = TOXACOL_PARQUET_DIR / "ToxACoL_All.parquet"

TOXACOL_COLUMNS = ["SMILES", "pred_mouse_intraperitoneal_LD50", "smiles_valid"]
TARGET_SCHEMA = pa.schema([
    pa.field("SMILES", pa.large_string()),
    pa.field("pred_mouse_intraperitoneal_LD50", pa.float64()),
    pa.field("smiles_valid", pa.bool_()),
])


def discover_batch_parquets(input_dir: Path, output_path: Path, require_exists: bool = False) -> list[Path]:
    if not input_dir.exists():
        if require_exists:
            raise FileNotFoundError(f"Input directory does not exist: {input_dir}")
        return []
    files = []
    for path in sorted(input_dir.glob("Batch*.parquet")):
        if path.resolve() == output_path.resolve():
            continue
        if path.name.endswith(".tmp.parquet"):
            continue
        files.append(path)
    return files


def validate_parquet_schema(path: Path) -> None:
    schema = pq.ParquetFile(path).schema_arrow
    missing = [column for column in TOXACOL_COLUMNS if column not in schema.names]
    extra = [name for name in schema.names if name not in TOXACOL_COLUMNS]
    if missing:
        raise ValueError(f"{path.name} is missing columns: {missing}")
    if extra:
        raise ValueError(f"{path.name} has unexpected columns: {extra}")


def align_table(table: pa.Table) -> pa.Table:
    arrays = []
    for field in TARGET_SCHEMA:
        array = table[field.name]
        if array.type != field.type:
            array = pc.cast(array, field.type)
        arrays.append(array)
    return pa.Table.from_arrays(arrays, schema=TARGET_SCHEMA)


def merge_toxacol_parquets(input_paths: list[Path], output_path: Path, overwrite: bool = False) -> dict:
    if not input_paths:
        raise FileNotFoundError(f"No Batch*.parquet files found in {TOXACOL_PARQUET_DIR}")
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"Output already exists: {output_path}. Set OVERWRITE_ALL_OUTPUT = True to replace it.")

    for path in input_paths:
        validate_parquet_schema(path)

    temp_path = output_path.with_name(output_path.stem + ".tmp.parquet")
    if temp_path.exists():
        if overwrite:
            temp_path.unlink()
        else:
            raise FileExistsError(f"Temporary output already exists: {temp_path}")

    writer = None
    rows_written = 0
    row_groups_written = 0
    try:
        writer = pq.ParquetWriter(
            temp_path,
            TARGET_SCHEMA,
            compression="snappy",
            use_dictionary=["SMILES"],
        )
        file_iter = tqdm(input_paths, desc="Merge ToxACoL files", unit="file", dynamic_ncols=True)
        for path in file_iter:
            parquet_file = pq.ParquetFile(path)
            file_iter.set_postfix(file=path.name, rows=f"{parquet_file.metadata.num_rows:,}")
            for row_group_index in range(parquet_file.metadata.num_row_groups):
                table = parquet_file.read_row_group(row_group_index, columns=TOXACOL_COLUMNS)
                table = align_table(table)
                writer.write_table(table)
                rows_written += table.num_rows
                row_groups_written += 1
    finally:
        if writer is not None:
            writer.close()

    if rows_written == 0:
        temp_path.unlink(missing_ok=True)
        raise RuntimeError("No rows were written; merge aborted.")

    if output_path.exists() and overwrite:
        output_path.unlink()
    temp_path.replace(output_path)
    return {
        "output_path": str(output_path),
        "rows_written": rows_written,
        "row_groups_written": row_groups_written,
        "size_gib": output_path.stat().st_size / 1024**3,
    }


batch_parquets = discover_batch_parquets(TOXACOL_PARQUET_DIR, ALL_OUTPUT_PATH, require_exists=RUN_MERGE)
print(f"Batch parquet dir: {TOXACOL_PARQUET_DIR}")
print(f"Output path:       {ALL_OUTPUT_PATH}")
print(f"Input files:       {len(batch_parquets)}")

if RUN_MERGE:
    result = merge_toxacol_parquets(batch_parquets, ALL_OUTPUT_PATH, overwrite=OVERWRITE_ALL_OUTPUT)
    print("Merge complete.")
    for key, value in result.items():
        print(f"{key}: {value}")
else:
    total_rows = 0
    for path in batch_parquets:
        metadata = pq.ParquetFile(path).metadata
        total_rows += metadata.num_rows
        print(f"{path.name:20s} rows={metadata.num_rows:>12,} row_groups={metadata.num_row_groups:>5,}")
    print("-" * 80)
    print("Dry run only. Set RUN_MERGE = True to create the merged parquet file.")
    print(f"Expected total rows: {total_rows:,}")


Batch parquet dir: C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet
Output path:       C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\ToxACoL_All.parquet
Input files:       22


Merge ToxACoL files:   0%|          | 0/22 [00:00<?, ?file/s]

Merge complete.
output_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\ToxACoL_All.parquet
rows_written: 8716446
row_groups_written: 1748
size_gib: 0.24730935785919428


## Deduplicate Merged ToxACoL Parquet

Run this cell to deduplicate any ToxACoL parquet by `SMILES`. This cell is standalone and does not require running the previous cells. Set `RUN_DEDUP = True` before running.


In [5]:
from pathlib import Path
import math
import sqlite3

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

RUN_DEDUP = True

DEDUP_INPUT_PATH = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\ToxACoL_All.parquet")
DEDUP_OUTPUT_PATH = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\ToxACoL_All_dedup.parquet")

TOXACOL_COLUMNS = ["SMILES", "pred_mouse_intraperitoneal_LD50", "smiles_valid"]
TARGET_SCHEMA = pa.schema([
    pa.field("SMILES", pa.large_string()),
    pa.field("pred_mouse_intraperitoneal_LD50", pa.float64()),
    pa.field("smiles_valid", pa.bool_()),
])

DEDUP_COLUMNS = ["SMILES"]
DEDUP_BATCH_SIZE = 50_000
DEDUP_SQLITE_PATH = DEDUP_OUTPUT_PATH.with_name(DEDUP_OUTPUT_PATH.stem + "_seen.sqlite")
DEDUP_KEEP = "first"
OVERWRITE_DEDUP_OUTPUT = False
CLEAN_DEDUP_TEMP = True


def validate_dedup_settings(columns: list[str], keep: str, batch_size: int) -> None:
    if keep != "first":
        raise ValueError("Only DEDUP_KEEP = 'first' is supported for streaming deduplication.")
    if columns != ["SMILES"]:
        raise ValueError('This notebook is configured to deduplicate ToxACoL by ["SMILES"] only.')
    if batch_size < 1:
        raise ValueError("DEDUP_BATCH_SIZE must be at least 1.")


def validate_parquet_schema(path: Path, dedup_columns: list[str]) -> None:
    schema = pq.ParquetFile(path).schema_arrow
    missing_output_columns = [column for column in TOXACOL_COLUMNS if column not in schema.names]
    if missing_output_columns:
        raise ValueError(f"{path.name} is missing output columns: {missing_output_columns}")
    missing_dedup_columns = [column for column in dedup_columns if column not in schema.names]
    if missing_dedup_columns:
        raise ValueError(f"DEDUP_COLUMNS contains unknown columns: {missing_dedup_columns}")


def align_table(table: pa.Table) -> pa.Table:
    arrays = []
    for field in TARGET_SCHEMA:
        array = table[field.name]
        if array.type != field.type:
            array = pc.cast(array, field.type)
        arrays.append(array)
    return pa.Table.from_arrays(arrays, schema=TARGET_SCHEMA)


def _dedup_value(value):
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def _dedup_key(value_tuple: tuple) -> str:
    return repr(tuple(_dedup_value(value) for value in value_tuple))


def make_key_rows(table: pa.Table, dedup_columns: list[str]) -> list[tuple[int, str]]:
    key_columns = [table[column].to_pylist() for column in dedup_columns]
    return [(row_index, _dedup_key(values)) for row_index, values in enumerate(zip(*key_columns))]


def setup_seen_key_db(db_path: Path, overwrite: bool) -> sqlite3.Connection:
    if db_path.exists():
        if overwrite:
            db_path.unlink()
        else:
            raise FileExistsError(f"Temporary SQLite key DB already exists: {db_path}")
    conn = sqlite3.connect(str(db_path))
    conn.execute("PRAGMA journal_mode=OFF")
    conn.execute("PRAGMA synchronous=OFF")
    conn.execute("PRAGMA temp_store=FILE")
    conn.execute("PRAGMA cache_size=-200000")
    conn.execute("CREATE TABLE seen_keys (key TEXT PRIMARY KEY)")
    conn.execute("CREATE TEMP TABLE batch_keys (pos INTEGER NOT NULL, key TEXT NOT NULL)")
    conn.execute("CREATE INDEX batch_keys_key_pos_idx ON batch_keys(key, pos)")
    conn.commit()
    return conn


def find_new_positions(conn: sqlite3.Connection, key_rows: list[tuple[int, str]]) -> list[int]:
    conn.execute("DELETE FROM batch_keys")
    conn.executemany("INSERT INTO batch_keys(pos, key) VALUES (?, ?)", key_rows)
    keep_rows = conn.execute("""
        SELECT MIN(b.pos) AS pos, b.key
        FROM batch_keys b
        LEFT JOIN seen_keys s ON s.key = b.key
        WHERE s.key IS NULL
        GROUP BY b.key
    """).fetchall()
    if keep_rows:
        conn.executemany("INSERT OR IGNORE INTO seen_keys(key) VALUES (?)", [(key,) for _, key in keep_rows])
    conn.commit()
    return [pos for pos, _ in keep_rows]


def filter_positions(table: pa.Table, keep_positions: list[int]) -> pa.Table:
    if not keep_positions:
        return table.slice(0, 0)
    keep_positions.sort()
    return table.take(pa.array(keep_positions, type=pa.int64()))


def deduplicate_toxacol_parquet_sqlite(
    input_path: Path,
    output_path: Path,
    dedup_columns: list[str],
    batch_size: int,
    sqlite_path: Path,
    overwrite: bool = False,
    clean_temp: bool = True,
) -> dict:
    validate_dedup_settings(dedup_columns, DEDUP_KEEP, batch_size)
    validate_parquet_schema(input_path, dedup_columns)
    if input_path.resolve() == output_path.resolve():
        raise ValueError("DEDUP_OUTPUT_PATH must be different from DEDUP_INPUT_PATH.")
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"Dedup output already exists: {output_path}. Set OVERWRITE_DEDUP_OUTPUT = True to replace it.")

    temp_output_path = output_path.with_name(output_path.stem + ".tmp.parquet")
    if temp_output_path.exists():
        if overwrite:
            temp_output_path.unlink()
        else:
            raise FileExistsError(f"Temporary dedup output already exists: {temp_output_path}")

    conn = setup_seen_key_db(sqlite_path, overwrite=overwrite)
    parquet_file = pq.ParquetFile(input_path)
    writer = None
    rows_read = 0
    rows_written = 0
    duplicate_rows_skipped = 0
    batches_written = 0

    total_rows = parquet_file.metadata.num_rows
    progress = tqdm(
        parquet_file.iter_batches(batch_size=batch_size, columns=TOXACOL_COLUMNS),
        total=math.ceil(total_rows / batch_size),
        desc="Deduplicate ToxACoL parquet",
        unit="batch",
        dynamic_ncols=True,
    )

    try:
        writer = pq.ParquetWriter(
            temp_output_path,
            TARGET_SCHEMA,
            compression="snappy",
            use_dictionary=["SMILES"],
        )
        for record_batch in progress:
            table = pa.Table.from_batches([record_batch])
            rows_read += table.num_rows
            table = align_table(table)
            key_rows = make_key_rows(table, dedup_columns)
            keep_positions = find_new_positions(conn, key_rows)
            duplicate_rows_skipped += table.num_rows - len(keep_positions)
            if keep_positions:
                filtered = filter_positions(table, keep_positions)
                writer.write_table(filtered)
                rows_written += filtered.num_rows
                batches_written += 1
            progress.set_postfix(read=f"{rows_read:,}", written=f"{rows_written:,}", skipped=f"{duplicate_rows_skipped:,}")
    finally:
        progress.close()
        if writer is not None:
            writer.close()
        conn.close()

    if rows_written == 0:
        temp_output_path.unlink(missing_ok=True)
        raise RuntimeError("No rows were written; dedup aborted.")

    if output_path.exists() and overwrite:
        output_path.unlink()
    temp_output_path.replace(output_path)
    if clean_temp:
        sqlite_path.unlink(missing_ok=True)

    return {
        "input_path": str(input_path),
        "output_path": str(output_path),
        "dedup_columns": dedup_columns,
        "batch_size": batch_size,
        "rows_read": rows_read,
        "rows_written": rows_written,
        "duplicate_rows_skipped": duplicate_rows_skipped,
        "batches_written": batches_written,
        "size_gib": output_path.stat().st_size / 1024**3,
    }


if RUN_DEDUP:
    result = deduplicate_toxacol_parquet_sqlite(
        DEDUP_INPUT_PATH,
        DEDUP_OUTPUT_PATH,
        DEDUP_COLUMNS,
        DEDUP_BATCH_SIZE,
        DEDUP_SQLITE_PATH,
        overwrite=OVERWRITE_DEDUP_OUTPUT,
        clean_temp=CLEAN_DEDUP_TEMP,
    )
    print("Dedup complete.")
    for key, value in result.items():
        print(f"{key}: {value}")
else:
    print("Dry run only. Set RUN_DEDUP = True to create the deduplicated parquet file.")
    print(f"Dedup input:    {DEDUP_INPUT_PATH}")
    print(f"Dedup output:   {DEDUP_OUTPUT_PATH}")
    print(f"Dedup columns:  {DEDUP_COLUMNS}")


Deduplicate ToxACoL parquet:   0%|          | 0/175 [00:00<?, ?batch/s]

Dedup complete.
input_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\ToxACoL_All.parquet
output_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Origin\ToxACoL_Parquet\ToxACoL_All_dedup.parquet
dedup_columns: ['SMILES']
batch_size: 50000
rows_read: 8716446
rows_written: 8213623
duplicate_rows_skipped: 502823
batches_written: 175
size_gib: 0.22689152974635363
